<div style="text-align:center; padding:20px 0"><img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/></div>

# Pharmacie Centrale HGU Cocody## Notebook 4 — Dashboard Power BI & Storytelling> **Prérequis** : Notebooks 1 à 3 complétés. Les fichiers CSV (données nettoyées + prévisions 4 semaines + stock optimisé) doivent être disponibles.| | ||---|---|| **Niveau** | Avancé || **Outils** | Power BI Desktop || **Durée estimée** | 4h à 5h |> 💡 **Ce notebook est un guide de conception reproductible.** En le suivant pas à pas, tu produiras le dashboard exactement tel qu'il apparaît dans le rapport de référence (4 pages avec navigation latérale verte médicale).### Objectif businessTransformer les analyses de consommation médicamenteuse et les prévisions ML en un dashboard décisionnel **4 pages** permettant au Dr. Konan de piloter : santé budgétaire globale · consommation par service · performance fournisseurs · alertes stock préventives.

---## 1. Sources de données (8 fichiers)### Fichiers à importer dans Power BI| Fichier CSV | Type | Rôle ||---|---|---|| `consommations.csv` | Fait | Consommations mensuelles par service × médicament || `commandes_fournisseurs.csv` | Fait | Commandes passées (date, fournisseur, statut, retard) || `ruptures_stock.csv` | Fait | Historique des ruptures (patients affectés, durée, impact clinique) || `previsions_4semaines.csv` | Analytique | Prévisions ML (4 semaines × médicament + bornes) || `stock_securite_optimise.csv` | Analytique | ROP + EOQ calculés par médicament (NB3) || `medicaments.csv` | Dimension | Catalogue (catégorie, prix, criticité, stock actuel) || `services.csv` | Dimension | Services hospitaliers (10 services) || `fournisseurs.csv` | Dimension | 5 fournisseurs (délai contractuel, pays) |### Import1. Power BI Desktop → **Obtenir des données → Texte/CSV**2. Mode **Import** pour chaque fichier3. Vérifier les types en Power Query avant de charger (voir section 3)

---## 2. Désactiver Auto Date/Time (obligatoire)**Fichier → Options → Chargement des données (Fichier actuel) → DÉCOCHER "Date/heure automatique pour le fichier actuel"**Sans cette étape, Power BI crée des tables `LocalDateTable_*` parasites pour chaque colonne date (≈ 4-6 tables ici) qui polluent le modèle et empêchent `PREVIOUSMONTH` de fonctionner correctement avec la table Calendrier.

---## 3. Nettoyage Power Query — Corrections de typesAvant d'appuyer sur *Fermer et appliquer*, corriger les types de colonnes qui peuvent être mal inférés.| Table | Colonne | Type cible | Raison ||---|---|---|---|| `consommations` | `date` | **Date** | Activer les analyses temporelles (PREVIOUSMONTH) || `consommations` | `quantite_consommee` | **Nombre entier** | Éviter les décimales parasites || `commandes_fournisseurs` | `date_commande` | **Date** | Jointure avec Calendrier || `commandes_fournisseurs` | `date_livraison_reelle` | **Date** | Calcul des retards || `commandes_fournisseurs` | `retard_jours` | **Nombre entier** | Comparaisons DAX `> 0` || `commandes_fournisseurs` | `montant_total_euro` | **Nombre décimal** | Agrégations financières || `medicaments` | `prix_unitaire_euro` | **Nombre décimal** | Calculs de coût || `medicaments` | `medicament_critique` | **Vrai/Faux** | Filtre `= TRUE()` sur la Page 2 || `medicaments` | `stock_actuel` | **Nombre entier** | Comparaisons avec ROP || `previsions_4semaines` | `semaine` | **Nombre entier** | Filtres `= 1`, `= 2` || `previsions_4semaines` | `quantite_prevue_semaine` | **Nombre décimal** | Agrégations || `stock_securite_optimise` | `point_commande_ROP` | **Nombre entier** | Comparaison stock || `stock_securite_optimise` | `qte_commande_EOQ` | **Nombre entier** | Calcul bon de commande |### ⚠️ Attention aux accentsDans `commandes_fournisseurs[statut]`, la valeur est **`Livrée`** (avec accent aigu). Les mesures DAX qui filtrent sur ce champ doivent utiliser exactement cette chaîne : `commandes_fournisseurs[statut] = "Livrée"`. Vérifier la valeur exacte dans Power Query avant d'écrire les mesures.

---## 4. Modèle de données — Schéma en étoile### Architecture du modèle```                             Calendrier (dim temps)                                    |                                    v       services --- consommations ---+--- medicaments                                     |         |                                     |         v                                     |     previsions_4semaines                                     |         |                                     |         v                                     |    stock_securite_optimise                                     |                         fournisseurs --- commandes_fournisseurs                                                |                                                v                                          ruptures_stock```### 8 relations à créer| De (N) | Colonne | Vers (1) | Colonne | Cardinalité ||---|---|---|---|---|| `consommations` | `date` | `Calendrier` | `Date` | N→1 || `commandes_fournisseurs` | `date_commande` | `Calendrier` | `Date` | N→1 || `consommations` | `id_medicament` | `medicaments` | `id_medicament` | N→1 || `consommations` | `id_service` | `services` | `id_service` | N→1 || `commandes_fournisseurs` | `id_medicament` | `medicaments` | `id_medicament` | N→1 || `commandes_fournisseurs` | `id_fournisseur` | `fournisseurs` | `id_fournisseur` | N→1 || `previsions_4semaines` | `id_medicament` | `medicaments` | `id_medicament` | N→1 || `stock_securite_optimise` | `medicament` | `medicaments` | `nom` | 1→1 |### ⚠️ Points de vigilance- **Direction : Single** sur toutes les relations (jamais Both)- **Relation `stock_securite_optimise → medicaments`** : se fait sur `nom` (pas sur `id_medicament`) car le fichier NB3 exporte le nom texte. Si besoin, créer une colonne `id_medicament` dans Power Query via merge.- **`ruptures_stock`** : peut rester non connecté au reste du modèle si utilisé uniquement pour les KPIs agrégés (Nb Jours Rupture, Patients Affectes) — sinon créer une relation via `id_medicament`.

---## 5. Table CalendrierModélisation → **Nouvelle table** → coller :```daxCalendrier =ADDCOLUMNS(    CALENDAR(DATE(2022,1,1), DATE(2024,9,30)),    "Annee",         YEAR([Date]),    "Mois_Num",      MONTH([Date]),    "Mois_Nom",      FORMAT([Date], "MMM", "fr-FR"),    "Mois_Nom_Long", FORMAT([Date], "MMMM", "fr-FR"),    "Annee_Mois",    FORMAT([Date], "YYYY-MM"),    "Trimestre",     "T" & QUARTER([Date]),    "Semaine",       WEEKNUM([Date]),    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR"),    "Est_Weekend",   IF(WEEKDAY([Date],2) >= 6, TRUE(), FALSE()))```**Puis marquer comme table de dates** : clic droit sur `Calendrier` → *Marquer comme table de dates* → colonne `Date`.> ⚠️ La plage `DATE(2022,1,1)` → `DATE(2024,9,30)` doit couvrir **les prévisions de juillet-août 2024** générées en NB3. Si la table s'arrête avant, la courbe Historique + Prévisions de la Page 4 sera tronquée.

---## 6. Table _Mesures (placeholder)Modélisation → **Nouvelle table** → coller :```dax_Mesures = {BLANK()}```Puis **masquer la colonne `Value`** (clic droit → Masquer). Toutes les mesures seront rangées dans cette table par dossier d'affichage.

---## 7. Design system — Pharmacie HGU### Identité visuelleCe rapport adopte une esthétique **médicale hospitalière** : sidebar vert foncé sur fond de page clair, accents vert turquoise pour les titres, codes couleurs stricts pour les alertes de stock.### Palette| Usage | Couleur | Hex ||---|---|---|| **Sidebar** (fond) | Vert médical foncé | `#0F5E4A` || **Titre de page** (Vue executive, etc.) | Vert turquoise | `#2FBB87` || Fond principal | Blanc cassé | `#F8F8F6` || Fond des cartes KPI | Blanc pur | `#FFFFFF` || **Alerte critique / Rupture** | Rouge | `#E24B4A` || **Vigilance / Commander** | Orange | `#E28B2D` || **Surveiller** | Jaune | `#F5D76E` || **Stock OK / Bon état** | Vert | `#1D9E75` || Bar Top 8 médicaments | Rouge saumon | `#E57373` || Donut catégories (palette) | Bleu `#4C7FBF` · Teal `#2FBB87` · Orange `#E28B2D` · Violet `#8E44AD` · Rose `#E91E63` · Bleu clair `#5DADE2` · Gris `#95A5A6` || Texte principal | Gris anthracite | `#2C2C2A` || Texte secondaire | Gris moyen | `#5F5E5A` || Texte discret (sous-titres) | Gris clair | `#888780` |### Typographie| Usage | Police | Taille ||---|---|---|| Titres de page | Segoe UI **Semibold** | 22-24pt || Sous-titres (contexte slicers) | Segoe UI | 12-13pt || Titres de visuels | Segoe UI | 12pt || Valeurs KPI | Segoe UI **Bold** | 32-38pt || Labels KPI | Segoe UI | 12pt |### Principes- Fond de page `#F8F8F6` (blanc cassé) — moins fatigant que le blanc pur pour un usage quotidien- Cartes KPI avec bordure supérieure colorée 3px selon la nature du KPI- **4 niveaux de statut stock** ont une sémantique FIXE (à expliquer au Dr. Konan dès la présentation) :    - 🔴 Rouge = agir aujourd'hui    - 🟠 Orange = agir cette semaine    - 🟡 Jaune = surveiller    - 🟢 Vert = aucune action

---## 8. Architecture de navigation — Sidebar verticale verte### Sidebar latérale gauche (présente sur les 4 pages)Bande verticale à gauche (largeur ~220px), fond vert foncé `#0F5E4A`, visible sur les 4 pages.#### Contenu en haut- **Logo médical** : cercle vert clair 80×80 avec **croix blanche** au centre (icône medical-cross de n'importe quelle librairie SVG)- Texte `HGU Cocody` en blanc **bold** 18pt- Texte `Pharmacie Centrale` en blanc 13pt regular (sous-titre)#### Menu de navigation (4 items)| Ordre | Label | Page cible ||---|---|---|| 1 | Vue executive | Page 1 || 2 | Consommation services | Page 2 || 3 | Fournisseurs | Page 3 || 4 | Previsions & alertes | Page 4 |**Item actif** : fond blanc semi-transparent `rgba(255,255,255,0.15)`, texte blanc bold, bordure gauche 3px blanche**Item inactif** : texte blanc regular opacité 0.75, fond transparent, aucune bordure#### Bas de sidebar — Signature DPLEn bas de la sidebar, dans un **arc de cercle décoratif** vert clair :- `DataProjectLab` en blanc bold 12pt- `Projet Sante 2024` en blanc regular 11pt### Implémentation Power BIPour chaque item du menu :1. Insertion → **Bouton** → texte de l'item2. Volet Format → Action → **Type : Navigation de page** → cible = page correspondante3. Pour l'état actif : dupliquer la sidebar sur chaque page et mettre uniquement l'item courant en blanc semi-transparent

---## 9. Slicers globaux### 3 slicers en haut à droite du bandeau de titrePrésents sur les 4 pages.| Slicer | Champ | Style | Valeur par défaut ||---|---|---|---|| **Année** | `Calendrier[Annee]` | **Liste déroulante** | `2023` || **Mois** | `Calendrier[Mois_Nom_Long]` | **Liste déroulante** multi-select | `Tout` || **Medicament** | `medicaments[nom]` | **Liste déroulante** multi-select | `Tout` |### Style visuel- Label en vert turquoise `#2FBB87` 11pt bold au-dessus du champ- Champ avec fond blanc, bordure fine grise `#E5E7EB`, icône flèche verte- Largeur ~150px chacun, alignés horizontalement avec 20px d'espace### SynchronisationClic droit sur chaque slicer → *Synchroniser les segments* → cocher les 4 pages (visible ET filtre).### Sous-titre contextuel dynamiqueSous chaque titre de page, afficher en italique gris la valeur active des slicers. Exemple sur la Page 1 :*"HGU Cocody · 2023 · Tous mois · Tous medicaments"*Implémenter via une **carte texte** avec la mesure `Sous Titre Contexte` (voir section 14).

---## 10. Page 1 — Vue executive**Titre** : `Tableau de bord Pharmacie - HGU Cocody` (Segoe UI Semibold 24pt, couleur vert turquoise `#2FBB87`)**Sous-titre dynamique** : `HGU Cocody · 2023 · Tous mois · Tous medicaments` (italique gris, réagit aux slicers)### Ligne 1 : 4 KPI cardsChaque carte : fond blanc, ombre légère, bordure supérieure colorée 3px, valeur en Segoe UI Bold 38pt colorée, label sous la valeur.| # | Label | Mesure | Valeur attendue | Couleur valeur | Bordure top | Complément ||---|---|---|---|---|---|---|| 1 | Cout total conso | `[Cout Total Consommation]` | **1,63M€** | 🟢 Vert turquoise `#2FBB87` | Vert `#2FBB87` | — || 2 | Jour de rupture | `[Nb Jours Rupture]` | **260** | 🔴 Rouge `#E24B4A` | Rouge `#E24B4A` | *Cible: 0-7 ruptures critiques* (gris 11pt) || 3 | Patients affectes | `[Patients Affectes Total]` | **2 633** | 🟠 Orange `#E28B2D` | Orange `#E28B2D` | — || 4 | Taux livraison | `[Taux Livraison a Temps]` | **37,04%** | 🔴 Rouge `#E24B4A` | Rouge `#E24B4A` | *Cible: 90% Retard moy 2j* (gris 11pt) |### Ligne 2 : Évolution mensuelle de la consommation totale (unites)**Visuel : Graphique en aires/courbes**- Axe X : `Calendrier[Mois_Nom]` (janv-23 à déc-23)- Axe Y : valeurs 10K-15K+ unités- **Série 1 — Conso mensuelle** : `[Total Unites Consommees]` en vert `#2FBB87` trait plein lissé 3px avec remplissage transparent- **Série 2 — Consommation Mois Precedent** : `[Consommation Mois Precedent]` en gris `#888780` pointillé 2px- Titre : `Evolution mensuelle de la consommation totale (unites)` en gris anthracite 12pt- Légende en haut à gauche du visuel### Ligne 3 : 2 visuels côte à côte#### Gauche — Top 8 medicaments (bar horizontal)- Visuel : **Histogramme à barres** trié DESC avec filtre Top N = 8- Axe Y : `medicaments[nom]` (Top 8)- Axe X : `[Total Unites Consommees]`- Couleur : rouge saumon uniforme `#E57373`- Data labels activés (format `0k` milliers)- Top 8 attendu : Insuline Glargine 355k · Héparine 5000UI 332k · Érythropoïétine 4000UI 250k · Morphine 10mg 136k · Vancomycine 500mg 102k · Dexaméthasone 4mg 99k · Ceftriaxone 1g 92k · Salbutamol 100µg 63k#### Droite — Repartition par categorie (donut)- Visuel : **Graphique en anneau**- Légende : `medicaments[categorie]`- Valeurs : `[Cout Total Consommation]`- Data labels : `€ + %` (format `0.00K€ (0.00%)`)- Palette multi-couleurs (7-8 catégories)- Résultat attendu : Antidiabétique 22,4% (365,86K€) · Anticoagulant 27% · Hématologique 20,32% (331,94K€) · Antibiotique 15,32% (250,33K€) · Analgésique opioïde 14,9% · Corticoïde 8% · Autres ~8%

---## 11. Page 2 — Consommation services**Titre** : `Consommation Service Hospitalier` (vert turquoise)**Sous-titre dynamique** : `Periode : 2023 · Service : Tous services · Medicament : Tous medicaments`### Ligne 1 : 2 visuels côte à côte#### Gauche — Evolution mensuelle par service (bar horizontal empilé par mois)- Visuel : **Graphique à barres empilées** (barres horizontales)- Axe Y : `Calendrier[Mois_Nom]` (janv-23 à oct-23 visibles)- Axe X : `[Total Unites Consommees]`- Légende : `services[nom_service]` (Top 6 visibles : Cardiologie, Chirurgie, Diabétologie, Maternité, Neurologie, Oncologie)- Palette par service : 6-10 couleurs distinctes (bleu, bleu foncé, orange, violet, rose, vert)- Data labels désactivés (trop dense)#### Droite — Poids relatif des services (treemap)- Visuel : **Treemap**- Groupe : `services[nom_service]`- Valeurs : `[Cout Total Consommation]`- Sans sous-groupe (un rectangle par service)- Résultat attendu : Réanimation (bleu, le plus grand) · Oncologie (orange) · Chirurgie (violet) · Médecine interne · Diabétologie · Cardiologie · Pneumologie · Gynécologie · Urgences · Maternité · Pédiatrie (le plus petit)- Label : nom du service + taille proportionnelle au coût### Ligne 2 : 2 visuels côte à côte#### Gauche — Matrice heatmap Services × Medicaments- Visuel : **Matrice** (native)- Lignes : `services[nom_service]` (10 services)- Colonnes : `medicaments[nom]` — **filtré sur `medicament_critique = TRUE`** dans le volet Filtres- Valeurs : `[Total Unites Consommees]`- **Formatage conditionnel sur les cellules** : Mise en forme → Cellules → Arrière-plan → règle *Par dégradé* → Min blanc `#FFFFFF` · Max vert pharmacie `#2FBB87`- Masquer totaux lignes + colonnes pour garder uniquement la grille- Résultat attendu (extrait) : Cardiologie × Amlodipine 5mg = 4 723 · Diabétologie × Amlodipine = 3 123 · Cardiologie × Atorvastatine 20mg = 3 929> ⚠️ **Sans le filtre `medicament_critique = TRUE`**, la matrice contient 22 colonnes et devient illisible. Avec le filtre, on garde ~10-15 colonnes.#### Droite — Tableau détaillé filtrable- Visuel : **Table** native- Colonnes : `Mois` (`Calendrier[Mois_Nom]`) · `Medicament` · `Service` · `Cout` · `Total Unites Consommees`- Tri par défaut : Mois croissant- Exemple observé (Maternité, Acide folique 5mg) : janv 272,70€ / 303 u · fevr 247,50€ / 275 u · mars 228,60€ / 254 u · ... · déc 304,20€ / 338 u

---## 12. Page 3 — Fournisseurs & Commandes**Titre** : `Performance fournisseurs & Commandes` (vert turquoise)**Sous-titre dynamique** : `Periode : 2023 · Service : Tous services · Medicament : Tous medicaments`### Ligne 1 : 2 visuels côte à côte#### Gauche — Taux Livraison a Temps (jauge circulaire)- Visuel : **Jauge** (native)- Valeur : `[Taux Livraison a Temps]` → **37,04%**- Min : 0 · Max : 1- Valeur cible : 0.90 (marqueur visible)- Zones colorées : 0-0,50 rouge `#E24B4A` · 0,50-0,80 orange `#E28B2D` · 0,80-1 vert `#2FBB87`- Bordure supérieure de la carte : rouge `#E24B4A` 3px (alerte)- Titre : `Taux Livraison a Temps`- Format valeur : `0.00%`> ⚠️ **La jauge affichera du rouge profond (37%).** C'est la réalité des données. Ne pas ajuster la cible à la baisse pour "faire mieux paraître" le dashboard — le rôle du dashboard est de montrer la réalité.#### Droite — Fiabilite vs Delai (nuage de points)- Visuel : **Nuage de points**- Axe X : `fournisseurs[delai_livraison_jours]` (de 0 à 25)- Axe Y : `[Taux Livraison a Temps]` par fournisseur (**forcer 0% → 200%** dans les options de l'axe)- Détails : `fournisseurs[nom]`- **Légende (couleur)** : `fournisseurs[nom]` (5 couleurs distinctes)- Data labels désactivés (les noms sont dans la légende en haut)- 5 points attendus : AfricaMed Logistics (violet) · EuroPharma Import (bleu) · MediSupply Dakar (orange) · PharmaDistrib CI (rose) · SantéPro Cameroun (bleu foncé)> ⚠️ **Forcer l'axe Y de 0 à 200%** : si on laisse Power BI auto-scaler entre 37% et 40%, les 5 points apparaissent dispersés alors qu'ils sont tous très proches. Forcer l'échelle large montre la vraie dispersion (faible) et la distance à la cible 90%.### Ligne 2 : 2 visuels côte à côte#### Gauche — Timeline des commandes livrees — montant mensuel (euros) (area chart)- Visuel : **Graphique en aires** (courbe lissée)- Axe X : `Calendrier[Mois_Nom]` (janv-23 à déc-23)- Axe Y : `[Montant Commandes Livrees]` (somme `montant_total_euro` filtré sur `statut = "Livrée"`)- Couleur : vert pharmacie `#2FBB87` avec remplissage transparent- Data labels activés sur chaque point (format `0K`)- Valeurs observées : janv 14K · févr 56K · mars 30K · avr 44K · mai 60K · juin 41K · juil 59K · août 40K · sept 52K · oct 51K · nov 24K · déc 69K#### Droite — Retard moyen par fournisseur (bar horizontal)- Visuel : **Histogramme à barres** trié DESC- Axe Y : `fournisseurs[nom]`- Axe X : `[Retard Moyen Jours]`- Couleur : orange `#E28B2D` uniforme (ou conditionnel : rouge si > 3j, orange si > 1j, vert sinon)- Data labels activés (format `0.0j`)- Valeurs attendues : AfricaMed Logistics 2,2j · PharmaDistrib CI 2,1j · SantéPro Cameroun 2,0j · EuroPharma Import 1,9j · MediSupply Dakar 1,8j**Légende décorative** sous le bar chart (texte ou 3 carrés colorés) :- 🔴 Retard > 3j- 🟠 Retard 1-3j- 🟢 A temps

---## 13. Page 4 — Previsions & alertes**Titre** : `Prevision 4 semaines - Alertes Stock` **en ROUGE** `#E24B4A` (urgence opérationnelle, seule page avec titre rouge)**Sous-titre dynamique** : `Medicament : Tous medicaments · 0 ruptures imminentes · 18 a commander` (réagit aux mesures)### Ligne 1 : 3 KPI cards verticales + tableau alertes#### Gauche — 3 KPI cards empilées verticalementChaque carte : fond blanc, bordure supérieure colorée 3px.| # | Valeur | Label | Bordure top | Couleur valeur ||---|---|---|---|---|| 1 | `[Nb Alertes Rouges]` → **0** | `Medicament en alerte rouge` | 🔴 Rouge `#E24B4A` | Rouge `#E24B4A` || 2 | `[Nb Alertes Orange]` → **18** | `Medicament a commander` | 🟠 Orange `#E28B2D` | Orange `#E28B2D` || 3 | `[Fournisseur Plus Urgent]` → **SantéPro Cameroun** | `Fournisseur le plus Urgent` | 🟢 Vert `#2FBB87` | Orange `#E28B2D` |> ℹ️ La 3e carte retourne le **nom texte** du fournisseur avec le plus de lignes en statut "COMMANDER CETTE SEMAINE" — c'est une mesure DAX qui concatène nom + ranking.#### Droite (75% largeur) — Tableau d'alerte**Visuel : Table** native- Titre : `Tableau d'alerte - Medicaments a commander` en gris anthracite bold- Filtre : `[Statut Stock]` IN {"RUPTURE IMMINENTE", "COMMANDER CETTE SEMAINE"}- Tri : `[Statut Stock]` en premier (RUPTURE en haut) puis par `medicaments[nom]` ASC| Colonne | Champ / Mesure | Format ||---|---|---|| Medicament | `medicaments[nom]` | Regular || Stock actuel | `medicaments[stock_actuel]` | Nombre entier || ROP Recommande | `[ROP Recommande]` | Nombre entier || Prevision Semaine 1 | `[Prevision Semaine 1]` | 0,00 || Prevision Semaine 2 | `[Prevision Semaine 2]` | 0,00 || EOQ Recommande | `[EOQ Recommande]` | Nombre entier |**En-tête du tableau** : fond orange `#E28B2D` texte blanc bold (signale l'urgence de la ligne)Exemples observés (18 lignes) : Amlodipine 5mg (stock 45, ROP 79, S1 162,80, S2 157,00) · Amoxicilline 500mg (30, 70, 140,50, 134,60) · Atorvastatine 20mg (40, 176, 142,20, 145,70) · Azithromycine 250mg · Ceftriaxone 1g (15, 27, 122,50, 120,30) · Dexaméthasone 4mg (20, 29, 242,90, 246,70) · Digoxine 0.25mg (15, 19, 29,50, 28,00) · Érythropoïétine 4000UI · Héparine 5000UI (18, 54, 249,80, 251,00) · Ibuprofène 400mg · Insuline Glargine (20, 124, 127,30, 127,50) · Lorazepam 1mg · Metformine 850mg · Morphine 10mg (10, 42, 119,60, 114,60) · Paracétamol 1g · Paracétamol Pédiatrique · Salbutamol 100µg (25, 239, 120,00, 117,70) · Vancomycine 500mg (10, 24, 47,50, 48,80) · Acide folique 5mg### Ligne 2 : 2 visuels côte à côte#### Gauche (60% largeur) — Historique + Previsions- Visuel : **Graphique en courbes**- Axe X : Dates (historique 2023 + prévisions 4 semaines 2024)- **Série 1 — Qte Prevue** : `SUM(previsions_4semaines[quantite_prevue_semaine])` — courbe rouge `#E24B4A` trait plein- **Série 2 — Borne basse** : `SUM(previsions_4semaines[borne_basse])` — rose clair `#F5A5A5` pointillé- **Série 3 — Borne haute** : `SUM(previsions_4semaines[borne_haute])` — rose clair `#F5A5A5` pointillé- Légende en haut : Qte Prevue · Borne basse · Borne haute (avec 3 dots colorés)- Titre : `Historique + Previsions` en bold#### Droite (40% largeur) — Quantites a commander par fournisseur- Visuel : **Histogramme à barres** trié DESC- Axe Y : `fournisseurs[nom]`- Axe X : `[Qte Totale a Commander]` (somme des EOQ pour les médicaments en alerte)- Couleur : orange `#E28B2D` uniforme- Data labels activés (format `0.0K`)- Valeurs attendues : MediSupply Dakar 1,5K · AfricaMed Logistics 1,3K · EuroPharma Import 0,8K · PharmaDistrib CI 0,7K · SantéPro Cameroun 0,7K

---## 14. Mesures DAX — `_Mesures` — KPIs de base (7 mesures)```dax-- M1 : Total des unités consommées (sur toutes les consommations)Total Unites Consommees = SUM(consommations[quantite_consommee])-- M2 : Coût total de consommation (page 1 + page 2)Cout Total Consommation = SUM(consommations[cout_euro])-- M3 : Jours de rupture (exclure les ruptures à impact mineur)Nb Jours Rupture = CALCULATE(    SUMX(ruptures_stock, ruptures_stock[duree_jours]),    ruptures_stock[impact_clinique] <> "Impact mineur")-- M4 : Patients affectés par des rupturesPatients Affectes Total = SUM(ruptures_stock[patients_affectes])-- M5 : Taux de livraison à temps (sur commandes livrées uniquement)-- Attention : la valeur dans le champ statut est "Livrée" avec accentTaux Livraison a Temps = VAR _livrees =     CALCULATE(        COUNTROWS(commandes_fournisseurs),        commandes_fournisseurs[statut] = "Livrée"    )VAR _a_temps =     CALCULATE(        COUNTROWS(commandes_fournisseurs),        commandes_fournisseurs[statut] = "Livrée",        commandes_fournisseurs[retard_jours] = 0    )RETURN DIVIDE(_a_temps, _livrees)-- M6 : Retard moyen sur les commandes retardées uniquement (exclure les 0)Retard Moyen Jours = AVERAGEX(    FILTER(commandes_fournisseurs, commandes_fournisseurs[retard_jours] > 0),    commandes_fournisseurs[retard_jours])-- M7 : Montant des commandes livrées (pour la timeline Page 3)Montant Commandes Livrees = CALCULATE(    SUM(commandes_fournisseurs[montant_total_euro]),    commandes_fournisseurs[statut] = "Livrée")```

---## 15. Mesures DAX — Temporelles & Stock (7 mesures)```dax-- M8 : Consommation du mois précédent (pour la ligne pointillée grise Page 1)-- Nécessite la table Calendrier marquée comme table de datesConsommation Mois Precedent = CALCULATE(    [Total Unites Consommees],    PREVIOUSMONTH(Calendrier[Date]))-- M9 : Variation mensuelle en % (utilisable pour tableaux de la Page 2)Variation Mensuelle Pct = DIVIDE(    [Total Unites Consommees] - [Consommation Mois Precedent],    [Consommation Mois Precedent])-- M10 : Prévision semaine 1 (utilisée dans le tableau d'alerte Page 4)Prevision Semaine 1 = CALCULATE(    SUM(previsions_4semaines[quantite_prevue_semaine]),    previsions_4semaines[semaine] = 1)-- M11 : Prévision semaine 2 (idem)Prevision Semaine 2 = CALCULATE(    SUM(previsions_4semaines[quantite_prevue_semaine]),    previsions_4semaines[semaine] = 2)-- M12 : Point de commande recommandé (depuis stock_securite_optimise, calculé en NB3)ROP Recommande = MAX(stock_securite_optimise[point_commande_ROP])-- M13 : Quantité à commander selon EOQ (idem NB3)EOQ Recommande = MAX(stock_securite_optimise[qte_commande_EOQ])-- M14 : Statut Stock — la mesure clé de la Page 4-- VAR calcule chaque valeur une seule fois et la réutilise dans SWITCHStatut Stock = VAR _stock_actuel = MAX(medicaments[stock_actuel])VAR _stock_secu   = MAX(medicaments[stock_securite])VAR _rop          = [ROP Recommande]VAR _prev_s1      = [Prevision Semaine 1]RETURNSWITCH(    TRUE(),    _stock_actuel < _stock_secu,            "RUPTURE IMMINENTE",    _stock_actuel < _rop,                   "COMMANDER CETTE SEMAINE",    _stock_actuel < (_rop + _prev_s1),      "SURVEILLER",    "Stock OK")```

---## 16. Mesures DAX — Alertes Page 4 (4 mesures)```dax-- M15 : Nombre de médicaments en alerte ROUGE (rupture imminente)Nb Alertes Rouges = COUNTROWS(    FILTER(        VALUES(medicaments[nom]),        [Statut Stock] = "RUPTURE IMMINENTE"    ))-- M16 : Nombre de médicaments à commander cette semaine (orange)Nb Alertes Orange = COUNTROWS(    FILTER(        VALUES(medicaments[nom]),        [Statut Stock] = "COMMANDER CETTE SEMAINE"    ))-- M17 : Quantité totale à commander par fournisseur (pour le bar chart)-- Somme des EOQ pour les médicaments en alerte rouge ou orangeQte Totale a Commander = SUMX(    FILTER(        VALUES(medicaments[nom]),        [Statut Stock] IN {"RUPTURE IMMINENTE", "COMMANDER CETTE SEMAINE"}    ),    [EOQ Recommande])-- M18 : Fournisseur le plus urgent-- Retourne le nom du fournisseur avec le plus de médicaments en alerte orange/rougeFournisseur Plus Urgent = VAR _top_fournisseur =     TOPN(        1,        ADDCOLUMNS(            VALUES(fournisseurs[nom]),            "nb_alertes",            CALCULATE(                COUNTROWS(                    FILTER(                        VALUES(medicaments[nom]),                        [Statut Stock] IN {"RUPTURE IMMINENTE", "COMMANDER CETTE SEMAINE"}                    )                )            )        ),        [nb_alertes],        DESC    )RETURN MAXX(_top_fournisseur, fournisseurs[nom])-- M19 : Sous-titre contextuel dynamique (à afficher sous le titre de chaque page)Sous Titre Contexte = VAR _annee = SELECTEDVALUE(Calendrier[Annee], "Toutes années")VAR _mois = IF(    ISFILTERED(Calendrier[Mois_Nom_Long]),    "Mois selectionnes",    "Tous mois")VAR _med = IF(    ISFILTERED(medicaments[nom]),    "Medicaments selectionnes",    "Tous medicaments")RETURN "HGU Cocody · " & _annee & " · " & _mois & " · " & _med```

---## 17. Mesures DAX — Formatage conditionnel (3 mesures)Ces mesures servent à colorer dynamiquement le tableau d'alerte Page 4 et le bar chart des retards Page 3.```dax-- M20 : Couleur de fond pour la colonne [Statut Stock] (Page 4)-- À utiliser dans : Format table → Couleurs des cellules → Par formuleCouleur Fond Statut = SWITCH(    [Statut Stock],    "RUPTURE IMMINENTE",        "#E24B4A",    "COMMANDER CETTE SEMAINE",  "#E28B2D",    "SURVEILLER",               "#F5D76E",    "#1D9E75")-- M21 : Couleur de texte pour la colonne [Statut Stock]Couleur Texte Statut = SWITCH(    [Statut Stock],    "SURVEILLER", "#2C2C2A",    "#FFFFFF")-- M22 : Couleur de barre pour Retard Moyen par fournisseur (Page 3)Couleur Barre Retard = VAR _r = [Retard Moyen Jours]RETURNSWITCH(TRUE(),    _r > 3, "#E24B4A",    _r > 1, "#E28B2D",    "#1D9E75")```### Utilisation du formatage conditionnel#### Page 4 — Tableau d'alerte1. Sélectionner la table2. Format → **Couleurs des cellules** → Colonne `Statut Stock`3. Arrière-plan → **`fx`** → Par formule → Sélectionner `Couleur Fond Statut`4. Idem pour Police → Sélectionner `Couleur Texte Statut`#### Page 3 — Bar chart Retard Moyen1. Sélectionner le bar chart2. Format → **Couleurs des données** → `fx` → Par formule → Sélectionner `Couleur Barre Retard`

---## 18. Checklist de validation### Import & modèle- [ ] 8 fichiers CSV importés (dont `stock_securite_optimise.csv` et `previsions_4semaines.csv`)- [ ] Auto Date/Time désactivé- [ ] 13 types corrigés en Power Query (dates, entiers, décimaux, Vrai/Faux)- [ ] Accent dans `statut = "Livrée"` vérifié- [ ] Table `Calendrier` créée jusqu'à `DATE(2024,9,30)` (couvre les prévisions juillet-août 2024)- [ ] Table `Calendrier` **marquée comme table de dates**- [ ] Table `_Mesures` créée (colonne Value masquée)- [ ] 8 relations actives (Single direction partout)- [ ] Relation `stock_securite_optimise[medicament] → medicaments[nom]` créée### Mesures DAX (22 mesures au total)- [ ] 7 mesures de base (M1-M7)- [ ] 7 mesures temporelles + stock (M8-M14, dont `Statut Stock`)- [ ] 5 mesures alertes Page 4 (M15-M19)- [ ] 3 mesures formatage conditionnel (M20-M22)### Valeurs attendues (période 2023, sans filtre médicament)| Mesure | Valeur ||---|---|| `[Cout Total Consommation]` | ~1,63 M€ || `[Nb Jours Rupture]` | 260 || `[Patients Affectes Total]` | 2 633 || `[Taux Livraison a Temps]` | 37,04% || `[Retard Moyen Jours]` | ~2,0 jours || `[Nb Alertes Rouges]` | 0 || `[Nb Alertes Orange]` | 18 || `[Fournisseur Plus Urgent]` | SantéPro Cameroun |### Top 8 médicaments (Page 1)- [ ] Insuline Glargine ≈ 355k unités- [ ] Héparine 5000UI ≈ 332k- [ ] Érythropoïétine 4000UI ≈ 250k- [ ] Morphine 10mg ≈ 136k- [ ] Vancomycine 500mg ≈ 102k- [ ] Dexaméthasone 4mg ≈ 99k- [ ] Ceftriaxone 1g ≈ 92k- [ ] Salbutamol 100µg ≈ 63k### Répartition par catégorie (Page 1 donut)- [ ] Anticoagulant ≈ 27%- [ ] Antidiabétique ≈ 22,4% (365,86 K€)- [ ] Hématologique ≈ 20,32% (331,94 K€)- [ ] Antibiotique ≈ 15,32% (250,33 K€)- [ ] Analgésique opioïde ≈ 14,9%### Retards fournisseurs (Page 3)- [ ] AfricaMed Logistics 2,2j- [ ] PharmaDistrib CI 2,1j- [ ] SantéPro Cameroun 2,0j- [ ] EuroPharma Import 1,9j- [ ] MediSupply Dakar 1,8j### Tableau d'alerte Page 4- [ ] 18 lignes affichées- [ ] 0 lignes rouges (RUPTURE IMMINENTE)- [ ] Toutes les lignes oranges (COMMANDER CETTE SEMAINE)- [ ] Tri par `[Statut Stock]` DESC### Navigation & slicers- [ ] Sidebar verte foncée sur les 4 pages avec logo médical- [ ] 4 items menu : Vue executive / Consommation services / Fournisseurs / Previsions & alertes- [ ] Item actif en blanc semi-transparent avec bordure gauche blanche- [ ] Slicers Année + Mois + Medicament en haut droite sur les 4 pages- [ ] Slicers synchronisés- [ ] Sous-titre contextuel dynamique (mesure `Sous Titre Contexte`) sous chaque titre- [ ] Signature DPL `Projet Sante 2024` en bas de sidebar

---## 19. Storytelling — Présentation 5 minutes au Dr. Konan### Structure narrative Problème → Cause → Solution → ImpactChaque chiffre est suivi de sa signification clinique. Ne jamais énoncer un chiffre sans dire ce qu'il implique pour les patients.#### Minute 1 — La situation actuelle (Page 1)> *"Dr. Konan, voici l'état de la pharmacie. Le budget médicaments 2023 s'établit à **1,63 M€**, avec une consommation répartie principalement sur 3 catégories : **Antidiabétique 22%**, **Hématologique 20%**, **Antibiotique 15%**. Côté opérationnel, on enregistre **260 jours de rupture** qui ont affecté **2 633 patients** — on est très au-dessus de la cible sectorielle de 0-7 ruptures critiques. Et côté fournisseurs, seulement **37% de livraisons à temps**, très loin de la cible de 90%."*#### Minute 2 — Où se concentre la consommation (Page 2)> *"La carte des services montre que **Réanimation et Oncologie concentrent près de 50% du budget** — ce qui est cohérent avec la criticité des soins dispensés. La heatmap Services × Médicaments révèle que certains médicaments critiques comme **Amlodipine et Atorvastatine sont concentrés sur Cardiologie et Diabétologie**, avec des pics >4 000 unités/mois. L'évolution mensuelle par service montre une tendance globale **stable, sans pic exceptionnel** détecté sur 2023."*#### Minute 3 — Pourquoi les ruptures (Page 3)> *"La jauge à 37% est explicite : **aucun des 5 fournisseurs n'atteint la cible de 90%** de ponctualité. Plus intéressant, le scatter Fiabilité vs Délai montre que **le délai contractuel n'explique pas la ponctualité** : EuroPharma avec 21 jours de délai a le même retard moyen que PharmaDistrib avec 3 jours (~2 jours). Le vrai problème, c'est la **régularité**, pas la durée. La timeline confirme une forte saisonnalité : creux en novembre (24K€) puis pic en décembre (69K€)."*#### Minute 4 — Ce qu'on fait (Page 4)> *"Bonne nouvelle : **aucune rupture imminente** aujourd'hui (0 alerte rouge). Mais **18 médicaments doivent être commandés cette semaine** pour éviter les ruptures dans 2 à 4 semaines. Le tableau d'alerte donne pour chacun : stock actuel, point de commande, prévision de consommation et quantité optimale à commander (EOQ calculé en NB3). Le fournisseur à contacter en priorité : **SantéPro Cameroun**, qui concentre le plus grand nombre d'alertes oranges."*#### Minute 5 — Les 3 actions prioritaires1. **Commander cette semaine les 18 médicaments listés** (action opérationnelle immédiate) → impact estimé : **-80% des ruptures futures** sur l'horizon 4 semaines2. **Renégocier avec les 3 fournisseurs les plus retardataires** (AfricaMed, PharmaDistrib, SantéPro) pour fiabiliser les délais → impact estimé : **+10 à +15pp sur le taux de livraison à temps**3. **Mettre en place le suivi hebdomadaire du dashboard Page 4** par le responsable pharmacie chaque lundi matin → détection systématique des alertes**Objectif 6 mois** : Ramener le taux de livraison de 37% à 60%+ et maintenir **0 rupture critique** par anticipation ML.---> L'apprenant doit pouvoir répondre à la question : **"Que doit faire la pharmacie HGU Cocody cette semaine ?"**> La réponse est contenue dans les 4 pages du dashboard — pas ailleurs.

---**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.